# 📈 EDA & Visualization
**ClimateTrend Analyzer** | Notebook 3 of 3

---
### Goal
In this notebook we will create all project visualizations:
1. 🌡️ Global temperature trend over time
2. 💨 CO₂ concentration over time
3. 🔵 CO₂ vs Temperature scatter
4. 🚨 Anomaly detection
5. 📅 Decade-wise average temperature
6. 🔥 Correlation heatmap
7. 🌧️ Rainfall trend
---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11

os.makedirs('../outputs/graphs', exist_ok=True)
print('Ready ✅')

In [ ]:
# Load cleaned data
df = pd.read_csv('../dataset/processed_data/climate_cleaned.csv')
print(f'Loaded: {df.shape}')
df.head()

---
## 📊 Chart 1: Global Temperature Anomaly Over Time

> **What it shows:** How Earth's average temperature has changed compared to a baseline (usually 1951–1980 average). Positive values = warmer than normal.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))

ax.plot(df['Year'], df['Temperature_Anomaly'],
        color='#78c1f3', alpha=0.5, linewidth=1, label='Annual Anomaly')

if 'Temp_RollingMean' in df.columns:
    ax.plot(df['Year'], df['Temp_RollingMean'],
            color='#e05c5c', linewidth=2.5, label='5-Year Rolling Mean')

ax.axhline(0, color='gray', linestyle='--', linewidth=0.8, label='Baseline (0°C)')
ax.fill_between(df['Year'], df['Temperature_Anomaly'], 0,
                where=(df['Temperature_Anomaly'] > 0),
                alpha=0.2, color='red', label='Above Baseline')

ax.set_title('🌡️ Global Temperature Anomaly Over Time', fontsize=14, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Temperature Anomaly (°C)')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/graphs/01_temperature_trend.png', bbox_inches='tight')
plt.show()

---
## 📊 Chart 2: CO₂ Concentration Over Time

In [ ]:
if 'CO2_ppm' in df.columns:
    fig, ax = plt.subplots(figsize=(13, 5))
    ax.fill_between(df['Year'], df['CO2_ppm'], alpha=0.35, color='#f4a261')
    ax.plot(df['Year'], df['CO2_ppm'], color='#e76f51', linewidth=2)
    ax.set_title('💨 Atmospheric CO₂ Concentration Over Time', fontsize=14, fontweight='bold')
    ax.set_xlabel('Year')
    ax.set_ylabel('CO₂ (ppm)')
    plt.tight_layout()
    plt.savefig('../outputs/graphs/02_co2_trend.png', bbox_inches='tight')
    plt.show()
else:
    print('CO2_ppm column not found – skipping')

---
## 📊 Chart 3: CO₂ vs Temperature Anomaly

> **What it shows:** As CO₂ rises, does temperature rise too? Each dot = one year. Color = year (darker = newer).

In [ ]:
if 'CO2_ppm' in df.columns:
    fig, ax = plt.subplots(figsize=(8, 6))
    sc = ax.scatter(df['CO2_ppm'], df['Temperature_Anomaly'],
                    c=df['Year'], cmap='YlOrRd', s=45, alpha=0.85, edgecolors='none')
    plt.colorbar(sc, ax=ax, label='Year')

    # Add trend line
    z = np.polyfit(df['CO2_ppm'].dropna(), df['Temperature_Anomaly'].dropna(), 1)
    p = np.poly1d(z)
    x_line = np.linspace(df['CO2_ppm'].min(), df['CO2_ppm'].max(), 100)
    ax.plot(x_line, p(x_line), 'b--', linewidth=1.5, label='Trend Line')

    ax.set_title('🔵 CO₂ vs Temperature Anomaly', fontsize=14, fontweight='bold')
    ax.set_xlabel('CO₂ Concentration (ppm)')
    ax.set_ylabel('Temperature Anomaly (°C)')
    ax.legend()
    plt.tight_layout()
    plt.savefig('../outputs/graphs/03_co2_vs_temperature.png', bbox_inches='tight')
    plt.show()

---
## 📊 Chart 4: Anomaly Detection

> **What it shows:** Red dots = years where temperature exceeded +0.5°C threshold. These are "extreme warming" events.

In [ ]:
if 'Anomaly_Flag' in df.columns:
    anomalies = df[df['Anomaly_Flag'] == 1]
    normal    = df[df['Anomaly_Flag'] == 0]

    fig, ax = plt.subplots(figsize=(13, 5))
    ax.scatter(normal['Year'], normal['Temperature_Anomaly'],
               color='#74c69d', s=25, alpha=0.7, label='Normal Year')
    ax.scatter(anomalies['Year'], anomalies['Temperature_Anomaly'],
               color='#d62828', s=60, zorder=5, label=f'Anomaly (>0.5°C) — {len(anomalies)} years')
    ax.axhline(0.5, color='orange', linestyle='--', linewidth=1.3, label='Threshold 0.5°C')
    ax.axhline(0,   color='gray',   linestyle='--', linewidth=0.8)

    ax.set_title('🚨 Temperature Anomaly Detection', fontsize=14, fontweight='bold')
    ax.set_xlabel('Year')
    ax.set_ylabel('Temperature Anomaly (°C)')
    ax.legend()
    plt.tight_layout()
    plt.savefig('../outputs/graphs/04_anomaly_detection.png', bbox_inches='tight')
    plt.show()

    print(f'\nAnomaly years detected: {sorted(anomalies["Year"].tolist())}')

---
## 📊 Chart 5: Decade-wise Average Temperature

In [ ]:
if 'Decade' in df.columns:
    decade_avg = df.groupby('Decade')['Temperature_Anomaly'].mean().reset_index()

    colors = ['#d62828' if v > 0 else '#4361ee' for v in decade_avg['Temperature_Anomaly']]

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(decade_avg['Decade'].astype(str), decade_avg['Temperature_Anomaly'],
                  color=colors, edgecolor='white', linewidth=0.8)
    ax.axhline(0, color='black', linewidth=0.9)

    # Add value labels on bars
    for bar, val in zip(bars, decade_avg['Temperature_Anomaly']):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.01, f'{val:.2f}',
                ha='center', va='bottom', fontsize=9)

    ax.set_title('📅 Average Temperature Anomaly by Decade', fontsize=14, fontweight='bold')
    ax.set_xlabel('Decade')
    ax.set_ylabel('Avg Anomaly (°C)')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig('../outputs/graphs/05_decade_avg.png', bbox_inches='tight')
    plt.show()

---
## 📊 Chart 6: Correlation Heatmap

In [ ]:
numeric_df = df.select_dtypes(include=[np.number]).drop(
    columns=['Anomaly_Flag', 'Decade'], errors='ignore'
)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(numeric_df.corr(), annot=True, fmt='.2f',
            cmap='coolwarm', linewidths=0.5, ax=ax,
            annot_kws={'size': 10})
ax.set_title('🔥 Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/graphs/06_correlation_heatmap.png', bbox_inches='tight')
plt.show()

---
## 📊 Chart 7: Rainfall Trend (if available)

In [ ]:
if 'Rainfall_mm' in df.columns:
    fig, ax = plt.subplots(figsize=(13, 4))
    ax.plot(df['Year'], df['Rainfall_mm'], color='#219ebc', linewidth=1.5)
    ax.fill_between(df['Year'], df['Rainfall_mm'], alpha=0.2, color='#219ebc')
    ax.set_title('🌧️ Annual Rainfall Over Time', fontsize=14, fontweight='bold')
    ax.set_xlabel('Year')
    ax.set_ylabel('Rainfall (mm)')
    plt.tight_layout()
    plt.savefig('../outputs/graphs/07_rainfall_trend.png', bbox_inches='tight')
    plt.show()
else:
    print('Rainfall_mm not in dataset – skipping.')

---
## ✅ All Visualizations Complete!

Graphs saved to `outputs/graphs/`:
- `01_temperature_trend.png`
- `02_co2_trend.png`
- `03_co2_vs_temperature.png`
- `04_anomaly_detection.png`
- `05_decade_avg.png`
- `06_correlation_heatmap.png`
- `07_rainfall_trend.png` *(if Rainfall_mm present)*

➡️ Next step: run `src/model.py` for predictions!